## Imports

In [14]:
%load_ext autoreload
%autoreload 2
# Internal import
import deep_learning_land_use_classification.config as dataset
import deep_learning_land_use_classification.evaluation as evaluation

import torch
from transformers import AutoImageProcessor, AutoModel

from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb
import torch.nn as nn

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
# Wandb initialization for experiment tracking

run = wandb.init(
    entity="sstaheli52-wageningen-university-and-research",
    project="DL_land-use-classification",
    dir=dataset.WANDB_DIR,
    config={
        "architecture": "dinov3-vitl16",
        "model_name": "facebook/dinov3-vitl16-pretrain-sat493m",
        "pretrained": True,
        "pretraining_dataset": "sat493m",
        "pretraining_source": "huggingface",
        "weights": "facebook/dinov3-vitl16-pretrain-sat493m",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "num_classes": dataset.num_classes,
    },
)

config = run.config

In [16]:
pretrained_model_name = "facebook/dinov3-vitl16-pretrain-sat493m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)

In [17]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        inputs = processor(images=image, return_tensors="pt") # converts image to tensor and normalizes it
        inputs = {k: v.squeeze(0) for k, v in inputs.items()} # remove batch dimension

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return inputs, label

In [18]:
# Create datasets and dataloaders
train_dataset = MultiLabelDataset(dataset.train_df, dataset.class_names, processor)
test_dataset  = MultiLabelDataset(dataset.test_df, dataset.class_names, processor)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)


In [19]:
# this is a simple wrapper around the DINO backbone to add a classification head on top
class DinoClassifier(torch.nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = torch.nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values) # get the output of the backbone
        features = outputs.pooler_output
        return self.classifier(features)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

backbone = AutoModel.from_pretrained(pretrained_model_name)

model = DinoClassifier(backbone, num_classes=config.num_classes).to(device)
model = model.to(device)


Using device: cuda


Loading weights: 100%|██████████| 415/415 [00:00<00:00, 3556.66it/s]


In [21]:
labels = dataset.train_df[dataset.class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only head
    lr=config.learning_rate,
    weight_decay=0.01
)

In [22]:
run.watch(model, log="parameters", log_freq=100)

### Train Model

In [23]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for inputs, labels in loader:
        pixel_values = inputs["pixel_values"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [24]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            pixel_values = inputs["pixel_values"].to(device)
            labels = labels.to(device)

            outputs = model(pixel_values)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
epochs = config.epochs

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics(
        model,
        test_loader,
        device,
        threshold=0.5
    )

    # Log metrics to Weights & Biases
    run.log({"train_loss": train_loss, "test_loss": test_loss})
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(dataset.class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))

    run.log({
        "train_loss": train_loss,
        "test_loss": test_loss,
        "macro/precision": p_macro,
        "macro/recall": r_macro,
        "macro/f1": f1_macro,
        "class_metrics": class_metrics,
    })
    

Epoch 1/5
Train Loss: 1.0624
Test Loss:  1.0011


AttributeError: 'dict' object has no attribute 'to'